
# Kuroshio Daily FINAL FIX2
## Strict causal blocked hindcast + train-only adaptive EOF(99%) + Reviewer baselines

本版本用于替代上一轮 blocked-CV 的最终论文统计结果。

### 为什么必须重跑？

上一版按 `target_max` 划分 test block。这样会出现：

```text
Fold 1 earliest test issue = 2024-06-10
but training labels extend to = 2024-07-27
```

也就是说，模型在“2024-06-10 起报”的测试样本上，已经使用了起报日之后的训练标签，
这不符合真正的 forward hindcast / unseen-time evaluation。

FIX2 改为严格因果规则：

```text
all training target dates
        <
validation issue start - purge
        <
all validation target dates
        <
test issue start - purge
```

因此：

- train/validation/test 的 forecast origins 严格按时间推进；
- 训练/验证的全部标签在下一阶段起报前已经“可获得”；
- 不再存在 test +30/+60 target 落入 training label period 的情况。

### 第二个修复：Daily EOF 不再固定 30 modes

月平均数据中 30 个 EOF 模态可解释 >99% 方差；
但上一轮 daily blocked folds 中，Branch 2 的 30 模态只解释约 88.6%–92.4%。

FIX2 改为：

> 每个 fold 仅用 training data 拟合 EOF，并自动保留达到 99% 累积解释方差所需的最少模态数。

这既避免 leakage，也不会因为沿用 monthly 的 30-mode 数字而人为丢失 daily variability。

### Reviewer baselines

继续比较：

- Persistence
- Climatology
- AR-EOF
- Simple-LSTM
- EOF-Conv-LSTM
- EOF-Conv-GRU
- EOF-TCN-LSTM
- EOF-TCN-GRU

神经网络仍为 5 seeds。

本轮输出写入新的：

```text
FINAL_RETRAIN_REVIEWER_FIX2/
```

不会覆盖上一轮结果。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, time, random, warnings, hashlib
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    FINAL_DATASET: str = "R1_FINAL_MANUAL22/dataset/daily_dual_branch_paths_FINAL.npz"
    FINAL_META: str = "R1_FINAL_MANUAL22/dataset/daily_dual_branch_meta_FINAL.csv"

    OUT_ROOT: str = "FINAL_RETRAIN_REVIEWER_FIX2"
    DATA_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/实验数据"
    CKPT_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/模型权重"
    FIG_ZH_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/图表/中文"
    FIG_EN_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/图表/英文"
    REPORT_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/报告"
    PRED_DIR: str = "FINAL_RETRAIN_REVIEWER_FIX2/预测结果"

    # 保持原论文6个月历史 -> 未来3个月的任务语义
    LOOKBACK_DAYS: int = 180
    LEAD_DAYS: tuple = (30, 60, 90)

    # 输入允许极短原始缺日；forecast targets禁止插值
    MAX_INPUT_GAP_DAYS: int = 2
    MAX_INPUT_MISSING_FRAC: float = 0.03

    # 原模型路径表示
    LEGACY_BRANCH1_31PTS: bool = True
    BRANCH1_DROP_FIRST: int = 12
    BRANCH2_MAX_LON: float = 145.0

    # blocked temporal CV
    N_FOLDS: int = 3
    # FIX2严格hindcast：显式定义连续、不重叠的 validation/test issue blocks
    STRICT_VAL_BLOCKS: tuple = (
        ("2024-07-20", "2024-08-02"),
        ("2024-08-03", "2024-08-16"),
        ("2024-08-17", "2024-08-30"),
    )
    STRICT_TEST_BLOCKS: tuple = (
        ("2024-11-08", "2024-11-21"),
        ("2024-11-22", "2024-12-05"),
        ("2024-12-06", "2024-12-25"),
    )

    VAL_BLOCK_DAYS: int = 14
    PURGE_DAYS: int = 7
    MIN_TRAIN_ANCHORS: int = 40
    MIN_VAL_ANCHORS: int = 12
    MIN_TEST_ANCHORS: int = 12

    # EOF
    EOF_EXPLAINED_TARGET: float = 0.99
    BRANCH1_MAX_MODES: int = 31
    BRANCH2_MAX_MODES: int = 277  # FIX2: 允许daily数据自动达到99%解释方差

    # AR-EOF
    AR_LAGS: int = 7
    AR_ALPHA_CANDIDATES: tuple = (0.01, 0.1, 1.0, 10.0, 100.0)

    # neural training
    SEEDS: tuple = (20, 21, 22, 23, 24)
    BATCH_SIZE: int = 32
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    MAX_EPOCHS_B1: int = 150
    MAX_EPOCHS_B2: int = 100
    EARLY_STOP_PATIENCE: int = 20
    LR_PATIENCE: int = 8
    LR_FACTOR: float = 0.5
    DROPOUT: float = 0.30

    # 运行控制
    FORCE_RETRAIN: bool = False
    TRAIN_FINAL_FROZEN_MODEL: bool = True
    FINAL_VAL_DAYS: int = 35
    USE_AMP: bool = True

    # 绘图
    FIG_DPI: int = 300
    REQUIRE_CJK_FONT: bool = True

    RANDOM_SEED: int = 20260926

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
DATASET_PATH = ROOT / cfg.FINAL_DATASET
META_PATH = ROOT / cfg.FINAL_META

OUT = ROOT / cfg.OUT_ROOT
DATA_DIR = ROOT / cfg.DATA_DIR
CKPT_DIR = ROOT / cfg.CKPT_DIR
FIG_ZH_DIR = ROOT / cfg.FIG_ZH_DIR
FIG_EN_DIR = ROOT / cfg.FIG_EN_DIR
REPORT_DIR = ROOT / cfg.REPORT_DIR
PRED_DIR = ROOT / cfg.PRED_DIR

for p in [OUT, DATA_DIR, CKPT_DIR, FIG_ZH_DIR, FIG_EN_DIR, REPORT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

# ===== 1. 环境 / GPU / 中文字体预检 =====
import importlib.util, platform, sys

required = [
    "numpy","pandas","scipy","sklearn",
    "matplotlib","torch"
]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    raise RuntimeError(
        "缺少依赖: " + ", ".join(missing) +
        "\n请在WSL环境中安装后重启Kernel。"
    )

import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_cjk_font():
    # 优先直接使用Windows字体，避免WSL中文乱码
    candidates = [
        "/mnt/c/Windows/Fonts/msyh.ttc",
        "/mnt/c/Windows/Fonts/msyhbd.ttc",
        "/mnt/c/Windows/Fonts/simhei.ttf",
        "/mnt/c/Windows/Fonts/simsun.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
    ]

    for p in candidates:
        if Path(p).exists():
            try:
                font_manager.fontManager.addfont(p)
                name = FontProperties(fname=p).get_name()
                return p, name
            except Exception:
                pass

    # 再扫描系统字体
    for p in font_manager.findSystemFonts():
        try:
            name = FontProperties(fname=p).get_name()
            low = name.lower()
            if any(k in low for k in [
                "noto sans cjk", "microsoft yahei",
                "simhei", "simsun", "wenquanyi"
            ]):
                return p, name
        except Exception:
            pass

    return None, None

CJK_FONT_PATH, CJK_FONT_NAME = find_cjk_font()
print("CJK font:", CJK_FONT_PATH, CJK_FONT_NAME)

if cfg.REQUIRE_CJK_FONT and CJK_FONT_NAME is None:
    raise RuntimeError(
        "没有检测到中文字体，因此停止绘图，避免生成中文乱码。\n"
        "WSL通常可以直接访问Windows字体：/mnt/c/Windows/Fonts/msyh.ttc\n"
        "若该文件不存在，可安装 fonts-noto-cjk 后重启Kernel。"
    )

def use_zh_font():
    matplotlib.rcdefaults()
    if CJK_FONT_NAME:
        matplotlib.rcParams["font.family"] = [CJK_FONT_NAME, "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams["pdf.fonttype"] = 42
    matplotlib.rcParams["ps.fonttype"] = 42

def use_en_font():
    matplotlib.rcdefaults()
    matplotlib.rcParams["font.family"] = ["DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams["pdf.fonttype"] = 42
    matplotlib.rcParams["ps.fonttype"] = 42

# 中文字体真正渲染 smoke test
use_zh_font()
fig, ax = plt.subplots(figsize=(4,2))
ax.text(0.5,0.5,"黑潮路径预测：中文字体测试",ha="center",va="center")
ax.axis("off")
font_smoke = REPORT_DIR / "中文字体渲染测试.png"
fig.savefig(font_smoke,dpi=140,bbox_inches="tight")
plt.close(fig)
print("中文字体渲染测试 ->", font_smoke)

print("tabulate optional:", importlib.util.find_spec("tabulate") is not None)


In [ ]:

# ===== 2. 读取最终人工QC后的日尺度数据 =====
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"未找到最终冻结数据集:\n{DATASET_PATH}\n"
        "请先在 MANUAL22 Notebook 中运行“冻结最终轨迹”和“生成FINAL双分支日尺度数据集”两个单元。"
    )

with np.load(DATASET_PATH, allow_pickle=False) as z:
    dates = pd.to_datetime(z["dates"])
    B1 = np.asarray(z["branch1_lon"], dtype=np.float32)
    LAT1 = np.asarray(z["branch1_target_lats"], dtype=np.float64)
    B2 = np.asarray(z["branch2_lat"], dtype=np.float32)
    LON2 = np.asarray(z["branch2_target_lons"], dtype=np.float64)
    src = np.asarray(z["source"]).astype(str) if "source" in z.files else np.array(["unknown"]*len(dates))

order = np.argsort(dates)
dates = dates[order]
B1 = B1[order]
B2 = B2[order]
src = src[order]

# 去重
dup = pd.Index(dates).duplicated(keep="last")
if dup.any():
    keep = ~dup
    dates, B1, B2, src = dates[keep], B1[keep], B2[keep], src[keep]

# 忠实保持原Branch1 31点表示
if cfg.LEGACY_BRANCH1_31PTS:
    if B1.shape[1] <= cfg.BRANCH1_DROP_FIRST:
        raise RuntimeError("Branch1维度不足，无法执行legacy 31-point crop")
    B1 = B1[:, cfg.BRANCH1_DROP_FIRST:]
    LAT1 = LAT1[cfg.BRANCH1_DROP_FIRST:]

# Branch2使用122~145E完整范围
mask2 = LON2 <= cfg.BRANCH2_MAX_LON + 1e-9
B2 = B2[:, mask2]
LON2 = LON2[mask2]

if np.isnan(B1).any() or np.isnan(B2).any():
    raise RuntimeError("FINAL dataset 内部仍有 NaN。请先修复冻结数据集。")

calendar_days = int((dates.max()-dates.min()).days+1)
coverage = len(dates)/calendar_days

print("date range:",dates.min(),"->",dates.max())
print("daily path N:",len(dates))
print("calendar coverage:",f"{coverage:.3%}")
print("Branch1:",B1.shape,"lat:",LAT1[0],"->",LAT1[-1])
print("Branch2:",B2.shape,"lon:",LON2[0],"->",LON2[-1])

audit = pd.DataFrame({
    "日期":dates,
    "来源":src,
})
audit.to_csv(
    DATA_DIR/"00_最终日尺度路径数据审计.csv",
    index=False, encoding="utf-8-sig"
)

source_counts = audit["来源"].value_counts().rename_axis("来源").reset_index(name="天数")
source_counts.to_csv(
    DATA_DIR/"00_路径来源统计.csv",
    index=False, encoding="utf-8-sig"
)
display(source_counts)


In [ ]:

# ===== 3. 构建180-day → +30/+60/+90 day anchors =====
full_dates = pd.date_range(dates.min(), dates.max(), freq="D")

def matrix_to_daily_df(mat):
    df = pd.DataFrame(
        mat,
        index=pd.DatetimeIndex(dates).normalize()
    )
    return df.reindex(full_dates)

df1 = matrix_to_daily_df(B1)
df2 = matrix_to_daily_df(B2)

present = pd.Series(False,index=full_dates)
present.loc[pd.DatetimeIndex(dates).normalize()] = True

def longest_missing_run(mask):
    best=cur=0
    for v in mask:
        if bool(v):
            cur+=1
            best=max(best,cur)
        else:
            cur=0
    return best

def get_input_window(df,issue):
    start = issue-pd.Timedelta(days=cfg.LOOKBACK_DAYS-1)
    win = df.loc[start:issue].copy()

    if len(win) != cfg.LOOKBACK_DAYS:
        return None

    row_missing = win.isna().any(axis=1).to_numpy()

    if row_missing.mean() > cfg.MAX_INPUT_MISSING_FRAC:
        return None
    if longest_missing_run(row_missing) > cfg.MAX_INPUT_GAP_DAYS:
        return None

    win = win.interpolate(
        method="time",
        limit=cfg.MAX_INPUT_GAP_DAYS,
        limit_area="inside"
    ).ffill(
        limit=cfg.MAX_INPUT_GAP_DAYS
    ).bfill(
        limit=cfg.MAX_INPUT_GAP_DAYS
    )

    if win.isna().any().any():
        return None

    return win.to_numpy(dtype=np.float32)

def actual_target(df,dt):
    if dt not in df.index:
        return None
    if not bool(present.loc[dt]):
        return None
    row = df.loc[dt]
    if row.isna().any():
        return None
    return row.to_numpy(dtype=np.float32)

anchors=[]

min_issue = full_dates.min()+pd.Timedelta(days=cfg.LOOKBACK_DAYS-1)
max_issue = full_dates.max()-pd.Timedelta(days=max(cfg.LEAD_DAYS))

for issue in pd.date_range(min_issue,max_issue,freq="D"):
    x1=get_input_window(df1,issue)
    x2=get_input_window(df2,issue)
    if x1 is None or x2 is None:
        continue

    tdates=[issue+pd.Timedelta(days=d) for d in cfg.LEAD_DAYS]
    y1=[actual_target(df1,d) for d in tdates]
    y2=[actual_target(df2,d) for d in tdates]

    if any(v is None for v in y1) or any(v is None for v in y2):
        continue

    anchors.append({
        "issue":issue,
        "target_dates":tdates,
        "target_max":max(tdates),
        "X1":x1,
        "X2":x2,
        "Y1":np.stack(y1).astype(np.float32),
        "Y2":np.stack(y2).astype(np.float32),
    })

print("valid 180-day anchors:",len(anchors))

if len(anchors) < 180:
    warnings.warn(
        f"180-day anchors只有 {len(anchors)} 个。"
        "脚本仍可运行，但blocked-CV稳定性会下降。"
    )

anchor_index = pd.DataFrame({
    "样本编号":np.arange(len(anchors)),
    "起报日期":[a["issue"] for a in anchors],
    "30天目标日期":[a["target_dates"][0] for a in anchors],
    "60天目标日期":[a["target_dates"][1] for a in anchors],
    "90天目标日期":[a["target_dates"][2] for a in anchors],
    "最晚目标日期":[a["target_max"] for a in anchors],
})
anchor_index.to_csv(
    DATA_DIR/"01_180天历史预测样本索引.csv",
    index=False, encoding="utf-8-sig"
)
display(anchor_index.head())


In [ ]:

# ===== 4. FIX2严格因果 3-fold blocked hindcast =====
# 原则：
# 1) training 的所有 target_max 必须早于 validation issue start - purge
# 2) validation 的所有 target_max 必须早于 test issue start - purge
# 3) test 以 forecast issue date 划分，而不是以 target_max 划分
# 4) 三个 test issue blocks 连续向后推进且互不重叠

if len(cfg.STRICT_VAL_BLOCKS) != cfg.N_FOLDS or len(cfg.STRICT_TEST_BLOCKS) != cfg.N_FOLDS:
    raise RuntimeError("STRICT_VAL_BLOCKS / STRICT_TEST_BLOCKS 与 N_FOLDS 不一致")

FOLDS=[]
fold_rows=[]
sample_rows=[]
leakage_rows=[]

for fi in range(cfg.N_FOLDS):
    val_start=pd.Timestamp(cfg.STRICT_VAL_BLOCKS[fi][0])
    val_end=pd.Timestamp(cfg.STRICT_VAL_BLOCKS[fi][1])

    test_start=pd.Timestamp(cfg.STRICT_TEST_BLOCKS[fi][0])
    test_end=pd.Timestamp(cfg.STRICT_TEST_BLOCKS[fi][1])

    # 截止条件保证所有训练标签在validation起报前已经可用
    train_target_cutoff = val_start - pd.Timedelta(days=cfg.PURGE_DAYS)

    # validation本身按issue date选取；其全部标签必须在test起报前可用
    val_target_cutoff = test_start - pd.Timedelta(days=cfg.PURGE_DAYS)

    tr=[]
    va=[]
    te=[]

    for i,a in enumerate(anchors):
        issue=pd.Timestamp(a["issue"])
        tmax=pd.Timestamp(a["target_max"])

        if tmax < train_target_cutoff:
            tr.append(i)

        elif (
            val_start <= issue <= val_end and
            tmax < val_target_cutoff
        ):
            va.append(i)

        elif test_start <= issue <= test_end:
            te.append(i)

    tr=np.asarray(tr,dtype=int)
    va=np.asarray(va,dtype=int)
    te=np.asarray(te,dtype=int)

    if len(tr)<cfg.MIN_TRAIN_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} train anchors不足: {len(tr)}")
    if len(va)<cfg.MIN_VAL_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} val anchors不足: {len(va)}")
    if len(te)<cfg.MIN_TEST_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} test anchors不足: {len(te)}")

    # -------- 严格泄漏审计 --------
    train_target_max=max(pd.Timestamp(anchors[i]["target_max"]) for i in tr)
    val_issue_min=min(pd.Timestamp(anchors[i]["issue"]) for i in va)
    val_target_max=max(pd.Timestamp(anchors[i]["target_max"]) for i in va)
    test_issue_min=min(pd.Timestamp(anchors[i]["issue"]) for i in te)

    check_train_to_val = train_target_max < (val_issue_min-pd.Timedelta(days=cfg.PURGE_DAYS))
    check_val_to_test = val_target_max < (test_issue_min-pd.Timedelta(days=cfg.PURGE_DAYS))

    # 所有实际target日期集合也要求互不重叠
    def target_date_set(indices):
        s=set()
        for j in indices:
            s.update(pd.Timestamp(x) for x in anchors[j]["target_dates"])
        return s

    tr_targets=target_date_set(tr)
    va_targets=target_date_set(va)
    te_targets=target_date_set(te)

    overlap_tr_va=len(tr_targets & va_targets)
    overlap_tr_te=len(tr_targets & te_targets)
    overlap_va_te=len(va_targets & te_targets)

    check_no_target_overlap = (
        overlap_tr_va==0 and
        overlap_tr_te==0 and
        overlap_va_te==0
    )

    if not (check_train_to_val and check_val_to_test and check_no_target_overlap):
        raise RuntimeError(
            f"Fold {fi+1} 时间泄漏审计失败: "
            f"train→val={check_train_to_val}, "
            f"val→test={check_val_to_test}, "
            f"overlap={overlap_tr_va}/{overlap_tr_te}/{overlap_va_te}"
        )

    f={
        "fold":fi+1,
        "train_idx":tr,
        "val_idx":va,
        "test_idx":te,

        "train_target_end":train_target_max,
        "val_issue_start":val_issue_min,
        "val_issue_end":max(pd.Timestamp(anchors[i]["issue"]) for i in va),
        "val_target_end":val_target_max,
        "test_issue_start":test_issue_min,
        "test_issue_end":max(pd.Timestamp(anchors[i]["issue"]) for i in te),
        "test_target_end":max(pd.Timestamp(anchors[i]["target_max"]) for i in te),
    }
    FOLDS.append(f)

    fold_rows.append({
        "折次":fi+1,
        "训练样本数":len(tr),
        "验证样本数":len(va),
        "测试样本数":len(te),
        "训练标签最晚日期":train_target_max,
        "验证起报开始":f["val_issue_start"],
        "验证起报结束":f["val_issue_end"],
        "验证标签最晚日期":val_target_max,
        "测试起报开始":f["test_issue_start"],
        "测试起报结束":f["test_issue_end"],
        "测试标签最晚日期":f["test_target_end"],
        "隔离天数":cfg.PURGE_DAYS,
    })

    leakage_rows.append({
        "折次":fi+1,
        "训练标签最晚日期":train_target_max,
        "验证最早起报日期":val_issue_min,
        "验证标签最晚日期":val_target_max,
        "测试最早起报日期":test_issue_min,
        "训练到验证严格因果":check_train_to_val,
        "验证到测试严格因果":check_val_to_test,
        "训练验证目标日期重叠数":overlap_tr_va,
        "训练测试目标日期重叠数":overlap_tr_te,
        "验证测试目标日期重叠数":overlap_va_te,
        "泄漏审计":"PASS",
    })

    for split,key in [("训练","train_idx"),("验证","val_idx"),("测试","test_idx")]:
        for idx in f[key]:
            sample_rows.append({
                "折次":fi+1,
                "集合":split,
                "样本编号":int(idx),
                "起报日期":anchors[idx]["issue"],
                "30天目标日期":anchors[idx]["target_dates"][0],
                "60天目标日期":anchors[idx]["target_dates"][1],
                "90天目标日期":anchors[idx]["target_dates"][2],
                "最晚目标日期":anchors[idx]["target_max"],
            })

fold_df=pd.DataFrame(fold_rows)
sample_df=pd.DataFrame(sample_rows)
leakage_df=pd.DataFrame(leakage_rows)

fold_df.to_csv(
    DATA_DIR/"02_严格分块后报折次定义.csv",
    index=False,encoding="utf-8-sig"
)
sample_df.to_csv(
    DATA_DIR/"02_严格分块后报逐样本划分.csv",
    index=False,encoding="utf-8-sig"
)
leakage_df.to_csv(
    DATA_DIR/"02_严格时序数据泄漏审计.csv",
    index=False,encoding="utf-8-sig"
)

display(fold_df)
display(leakage_df)

print("[STRICT TEMPORAL LEAKAGE AUDIT] ALL PASS")


In [ ]:

# ===== 5. Fold-specific train-only scaler + EOF =====
from sklearn.linear_model import Ridge

class BranchPreprocessor:
    def __init__(self,max_modes,explained_target=0.99):
        self.max_modes=max_modes
        self.explained_target=explained_target

    def fit(self,X_train):
        # X_train: [N,T,M], 只用train input
        flat=X_train.reshape(-1,X_train.shape[-1]).astype(np.float64)

        self.mu=flat.mean(axis=0)
        self.std=flat.std(axis=0)
        self.std[self.std<1e-6]=1.0

        Z=(flat-self.mu)/self.std
        # SVD EOF
        _,s,vt=np.linalg.svd(Z,full_matrices=False)
        variance=s**2
        ratio=variance/(variance.sum()+1e-12)
        cum=np.cumsum(ratio)

        need=int(np.searchsorted(cum,self.explained_target)+1)
        if self.max_modes is not None:
            need=min(need,self.max_modes)
        need=min(need,vt.shape[0])

        self.V=vt[:need].T.astype(np.float32)
        self.explained=float(cum[need-1])
        self.n_modes=need
        return self

    def standardize(self,A):
        return ((A-self.mu)/self.std).astype(np.float32)

    def destandardize(self,Z):
        return (Z*self.std+self.mu).astype(np.float32)

    def path_to_pc(self,A):
        Z=self.standardize(A)
        return np.matmul(Z,self.V).astype(np.float32)

    def pc_to_path(self,PC):
        Z=np.matmul(PC,self.V.T)
        return self.destandardize(Z)

    def neural_input(self,X):
        Z=self.standardize(X)
        D=np.zeros_like(Z)
        D[:,1:]=Z[:,1:]-Z[:,:-1]
        return np.concatenate([Z,D],axis=-1).astype(np.float32)

def stack_anchor(indices,key):
    return np.stack([anchors[i][key] for i in indices]).astype(np.float32)

def dates_for(indices):
    issues=np.array([anchors[i]["issue"] for i in indices],dtype="datetime64[D]")
    targets=np.array([
        [np.datetime64(d.date()) for d in anchors[i]["target_dates"]]
        for i in indices
    ])
    return issues,targets

def prepare_fold(f):
    tri=f["train_idx"]; vai=f["val_idx"]; tei=f["test_idx"]

    X1tr=stack_anchor(tri,"X1")
    X2tr=stack_anchor(tri,"X2")

    pp1=BranchPreprocessor(
        cfg.BRANCH1_MAX_MODES,
        cfg.EOF_EXPLAINED_TARGET
    ).fit(X1tr)

    pp2=BranchPreprocessor(
        cfg.BRANCH2_MAX_MODES,
        cfg.EOF_EXPLAINED_TARGET
    ).fit(X2tr)

    out={"pp1":pp1,"pp2":pp2}

    for split,idx in [("train",tri),("val",vai),("test",tei)]:
        X1=stack_anchor(idx,"X1")
        X2=stack_anchor(idx,"X2")
        Y1=stack_anchor(idx,"Y1")
        Y2=stack_anchor(idx,"Y2")
        issues,tdates=dates_for(idx)

        out[split]={
            "X1_raw":X1,
            "X2_raw":X2,
            "Y1_raw":Y1,
            "Y2_raw":Y2,

            "X1":pp1.neural_input(X1),
            "X2":pp2.neural_input(X2),

            "X1_pc":pp1.path_to_pc(X1),
            "X2_pc":pp2.path_to_pc(X2),

            "Y1_pc":pp1.path_to_pc(Y1),
            "Y2_pc":pp2.path_to_pc(Y2),

            "issues":issues,
            "target_dates":tdates,
        }

    return out

FOLD_DATA={}
preprocess_rows=[]

for f in FOLDS:
    fd=prepare_fold(f)
    FOLD_DATA[f["fold"]]=fd

    preprocess_rows.append({
        "折次":f["fold"],
        "分支1_EOF模态数":fd["pp1"].n_modes,
        "分支1_累计解释方差":fd["pp1"].explained,
        "分支2_EOF模态数":fd["pp2"].n_modes,
        "分支2_累计解释方差":fd["pp2"].explained,
    })

preprocess_df=pd.DataFrame(preprocess_rows)
preprocess_df.to_csv(
    DATA_DIR/"03_各折训练集EOF配置.csv",
    index=False, encoding="utf-8-sig"
)
display(preprocess_df)

# FIX2: daily正式实验要求两分支都真正达到99% training variance
if (preprocess_df["分支1_累计解释方差"] < cfg.EOF_EXPLAINED_TARGET-1e-6).any():
    raise RuntimeError("Branch1 EOF未达到99%解释方差")
if (preprocess_df["分支2_累计解释方差"] < cfg.EOF_EXPLAINED_TARGET-1e-6).any():
    raise RuntimeError(
        "Branch2 EOF未达到99%解释方差。"
        "请增加 BRANCH2_MAX_MODES；不要继续正式训练。"
    )

print("[EOF 99% TRAIN-ONLY GATE] ALL PASS")


In [ ]:

# ===== 6. 路径合并与公里尺度评估 =====
def merge_branches(b1_lon,b2_lat):
    b1_lon=np.asarray(b1_lon,float)
    b2_lat=np.asarray(b2_lat,float)

    lon1=b1_lon
    lat1=LAT1

    end_lon=float(lon1[-1])
    end_lat=float(lat1[-1])

    valid=np.where(LON2>end_lon)[0]
    if len(valid)==0:
        return lon1.copy(),lat1.copy()

    d=haversine_km(
        end_lon,end_lat,
        LON2[valid],b2_lat[valid]
    )
    k=int(valid[np.argmin(d)])

    lon=np.r_[lon1,LON2[k:]]
    lat=np.r_[lat1,b2_lat[k:]]
    return lon.astype(np.float32),lat.astype(np.float32)

def haversine_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2=map(np.asarray,(lon1,lat1,lon2,lat2))
    R=6371.0088
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    dp=p2-p1
    aa=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def pairwise_local_km(lon_a,lat_a,lon_b,lat_b):
    lat0=float(np.nanmean(np.r_[lat_a,lat_b]))
    lon0=float(np.nanmean(np.r_[lon_a,lon_b]))

    def xy(lon,lat):
        x=(np.asarray(lon)-lon0)*111.32*np.cos(np.deg2rad(lat0))
        y=(np.asarray(lat)-lat0)*111.32
        return np.column_stack([x,y])

    A=xy(lon_a,lat_a)
    B=xy(lon_b,lat_b)
    return np.sqrt(((A[:,None,:]-B[None,:,:])**2).sum(axis=2))

def cross_track_metrics(pred_lon,pred_lat,true_lon,true_lat):
    d=pairwise_local_km(pred_lon,pred_lat,true_lon,true_lat)

    # symmetric nearest-path distance
    a=d.min(axis=1)
    b=d.min(axis=0)
    x=np.r_[a,b]

    return {
        "横向误差平均值_km":float(np.mean(x)),
        "横向误差中位数_km":float(np.median(x)),
        "横向误差RMSE_km":float(np.sqrt(np.mean(x**2))),
        "横向误差P90_km":float(np.percentile(x,90)),
        "横向误差P95_km":float(np.percentile(x,95)),
        "横向误差最大值_km":float(np.max(x)),
    }

def evaluate_predictions(
    fold,model,seed,
    pred1,pred2,
    true1,true2,
    issues,target_dates
):
    rows=[]

    for n in range(len(pred1)):
        for j,lead in enumerate(cfg.LEAD_DAYS):
            plon,plat=merge_branches(pred1[n,j],pred2[n,j])
            tlon,tlat=merge_branches(true1[n,j],true2[n,j])

            m=cross_track_metrics(plon,plat,tlon,tlat)

            row={
                "折次":int(fold),
                "模型":model,
                "随机种子":seed,
                "起报日期":pd.Timestamp(issues[n]),
                "目标日期":pd.Timestamp(target_dates[n,j]),
                "提前期_天":int(lead),

                "分支1经度RMSE_度":float(np.sqrt(np.mean((pred1[n,j]-true1[n,j])**2))),
                "分支1经度MAE_度":float(np.mean(np.abs(pred1[n,j]-true1[n,j]))),
                "分支2纬度RMSE_度":float(np.sqrt(np.mean((pred2[n,j]-true2[n,j])**2))),
                "分支2纬度MAE_度":float(np.mean(np.abs(pred2[n,j]-true2[n,j]))),
                **m
            }
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:

# ===== 7. 三个统计/传统基线：Persistence / Climatology / AR-EOF =====
from sklearn.linear_model import Ridge

def persistence_predict(split):
    # 最后一个历史日直接持续到未来
    p1=np.repeat(split["X1_raw"][:,-1:, :],len(cfg.LEAD_DAYS),axis=1)
    p2=np.repeat(split["X2_raw"][:,-1:, :],len(cfg.LEAD_DAYS),axis=1)
    return p1.astype(np.float32),p2.astype(np.float32)

def fit_climatology(fd,branch):
    Y=fd["train"][f"Y{branch}_raw"]
    tdates=fd["train"]["target_dates"]

    global_mean=Y.mean(axis=0)
    month_map={}

    for j in range(len(cfg.LEAD_DAYS)):
        months=pd.to_datetime(tdates[:,j]).month.to_numpy()
        for m in range(1,13):
            ids=np.where(months==m)[0]
            if len(ids):
                month_map[(j,m)]=Y[ids,j].mean(axis=0)

    return {"global":global_mean,"month":month_map}

def climatology_predict(model,target_dates):
    N=len(target_dates)
    L=len(cfg.LEAD_DAYS)
    M=model["global"].shape[-1]
    out=np.zeros((N,L,M),dtype=np.float32)

    for n in range(N):
        for j in range(L):
            month=pd.Timestamp(target_dates[n,j]).month
            out[n,j]=model["month"].get(
                (j,month),
                model["global"][j]
            )
    return out

def ar_features(Xpc,lags):
    return Xpc[:,-lags:,:].reshape(len(Xpc),-1)

def fit_ar_branch(fd,branch):
    Xtr=ar_features(fd["train"][f"X{branch}_pc"],cfg.AR_LAGS)
    Ytr=fd["train"][f"Y{branch}_pc"].reshape(len(Xtr),-1)

    Xva=ar_features(fd["val"][f"X{branch}_pc"],cfg.AR_LAGS)
    Yva=fd["val"][f"Y{branch}_pc"].reshape(len(Xva),-1)

    best=None
    for alpha in cfg.AR_ALPHA_CANDIDATES:
        reg=Ridge(alpha=float(alpha))
        reg.fit(Xtr,Ytr)
        pred=reg.predict(Xva)
        mse=float(np.mean((pred-Yva)**2))
        if best is None or mse<best["mse"]:
            best={"alpha":float(alpha),"mse":mse,"model":reg}

    return best

def ar_predict(best,fd,split,branch,n_modes):
    X=ar_features(fd[split][f"X{branch}_pc"],cfg.AR_LAGS)
    p=best["model"].predict(X)
    return p.reshape(len(X),len(cfg.LEAD_DAYS),n_modes).astype(np.float32)


In [ ]:

# ===== 8. PyTorch 模型：Simple + 原4套Hybrid =====
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

class PerStepConvEncoder(nn.Module):
    def __init__(self,feature_dim,variant,branch,dropout):
        super().__init__()
        self.variant=variant
        self.feature_dim=int(feature_dim)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)

        if variant=="conv":
            c1=16 if branch==1 else 32
            c2=32 if branch==1 else 64
            self.conv1=nn.Conv1d(1,c1,3,padding=1)
            self.conv2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*self.feature_dim

        elif variant=="tcn":
            c=16 if branch==1 else 32
            k=3 if branch==1 else 5
            hidden=32 if branch==1 else 64

            self.k=k
            self.c=c
            self.conv1=nn.Conv1d(
                1,c,kernel_size=k,dilation=2,padding=0
            )
            self.bn1=nn.BatchNorm1d(c)

            # MATLAB: left padding=2, dilated valid conv
            effective=(k-1)*2+1
            conv_len=(self.feature_dim+2)-effective+1
            if conv_len<=0:
                raise RuntimeError("invalid TCN conv length")

            self.fc=nn.Linear(c*conv_len,hidden)
            self.out_dim=hidden
        else:
            raise ValueError(variant)

    def forward(self,x):
        B,T,F=x.shape
        if F!=self.feature_dim:
            raise RuntimeError(
                f"feature dim mismatch {F}!={self.feature_dim}"
            )

        z=x.reshape(B*T,1,F)

        if self.variant=="conv":
            z=self.drop(self.relu(self.conv1(z)))
            z=self.drop(self.relu(self.conv2(z)))
            z=z.flatten(1)
        else:
            z=torch.nn.functional.pad(z,(2,0))
            z=self.conv1(z)
            z=self.drop(self.relu(self.bn1(z)))
            z=z.flatten(1)

            if z.shape[1]!=self.fc.in_features:
                raise RuntimeError(
                    f"TCN flatten {z.shape[1]} != fc {self.fc.in_features}"
                )
            z=self.drop(self.relu(self.fc(z)))

        return z.reshape(B,T,-1)

class HybridNet(nn.Module):
    def __init__(self,feature_dim,n_modes,branch,conv_kind,rnn_kind):
        super().__init__()

        self.enc=PerStepConvEncoder(
            feature_dim,conv_kind,branch,cfg.DROPOUT
        )

        hidden = (
            (16 if branch==1 else 64)
            if conv_kind=="conv"
            else (32 if branch==1 else 64)
        )

        RNN=nn.LSTM if rnn_kind=="lstm" else nn.GRU
        self.rnn=RNN(
            self.enc.out_dim,
            hidden,
            batch_first=True
        )
        self.head=nn.Linear(
            hidden,
            len(cfg.LEAD_DAYS)*n_modes
        )
        self.n_modes=n_modes

    def forward(self,x):
        z=self.enc(x)
        o,_=self.rnn(z)
        return self.head(o[:,-1]).reshape(
            x.shape[0],len(cfg.LEAD_DAYS),self.n_modes
        )

class SimpleLSTM(nn.Module):
    def __init__(self,feature_dim,n_modes,branch):
        super().__init__()
        hidden=32 if branch==1 else 64
        self.rnn=nn.LSTM(
            feature_dim,hidden,batch_first=True
        )
        self.head=nn.Linear(
            hidden,len(cfg.LEAD_DAYS)*n_modes
        )
        self.n_modes=n_modes

    def forward(self,x):
        o,_=self.rnn(x)
        return self.head(o[:,-1]).reshape(
            x.shape[0],len(cfg.LEAD_DAYS),self.n_modes
        )

MODEL_SPECS=[
    ("Simple-LSTM","simple","lstm"),
    ("EOF-Conv-LSTM","conv","lstm"),
    ("EOF-Conv-GRU","conv","gru"),
    ("EOF-TCN-LSTM","tcn","lstm"),
    ("EOF-TCN-GRU","tcn","gru"),
]

def build_model(name,conv_kind,rnn_kind,branch,fd):
    feature_dim=fd["train"][f"X{branch}"].shape[-1]
    n_modes=fd[f"pp{branch}"].n_modes

    if conv_kind=="simple":
        return SimpleLSTM(feature_dim,n_modes,branch)

    return HybridNet(
        feature_dim,n_modes,branch,
        conv_kind,rnn_kind
    )

# architecture preflight
preflight=[]
for fold,fd in FOLD_DATA.items():
    for name,conv_kind,rnn_kind in MODEL_SPECS:
        for branch in (1,2):
            m=build_model(name,conv_kind,rnn_kind,branch,fd)
            X=fd["train"][f"X{branch}"][:2]
            xb=torch.from_numpy(X).float()
            with torch.no_grad():
                y=m(xb)
            expected=(len(xb),len(cfg.LEAD_DAYS),fd[f"pp{branch}"].n_modes)
            if tuple(y.shape)!=expected:
                raise RuntimeError(
                    f"preflight fail fold={fold} model={name} branch={branch}: "
                    f"{tuple(y.shape)} != {expected}"
                )
            preflight.append({
                "折次":fold,
                "模型":name,
                "分支":branch,
                "输入形状":str(tuple(xb.shape)),
                "输出形状":str(tuple(y.shape)),
                "参数量":sum(p.numel() for p in m.parameters())
            })

preflight_df=pd.DataFrame(preflight)
preflight_df.to_csv(
    DATA_DIR/"04_神经网络结构预检.csv",
    index=False,encoding="utf-8-sig"
)
print("[architecture preflight] ALL PASS")


In [ ]:

# ===== 9. 训练 / checkpoint / resume =====
from contextlib import nullcontext

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_dir(fold,model,seed):
    p=CKPT_DIR/f"第{fold}折"/model/f"种子{seed}"
    p.mkdir(parents=True,exist_ok=True)
    return p

def safe_save_model(model,path,meta):
    torch.save({
        "state_dict":{
            k:v.detach().cpu()
            for k,v in model.state_dict().items()
        },
        "meta":meta
    },path)

def train_one(model,fd,branch,max_epochs,seed):
    seed_all(seed)
    model=model.to(device)

    Xtr=torch.from_numpy(fd["train"][f"X{branch}"]).float()
    Ytr=torch.from_numpy(fd["train"][f"Y{branch}_pc"]).float()
    Xva=torch.from_numpy(fd["val"][f"X{branch}"]).float()
    Yva=torch.from_numpy(fd["val"][f"Y{branch}_pc"]).float()

    train_loader=DataLoader(
        TensorDataset(Xtr,Ytr),
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    val_loader=DataLoader(
        TensorDataset(Xva,Yva),
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=cfg.LR_FACTOR,
        patience=cfg.LR_PATIENCE,
        min_lr=1e-5
    )
    loss_fn=nn.MSELoss()

    amp_enabled=bool(
        cfg.USE_AMP and
        torch.cuda.is_available()
    )
    scaler=torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled
    )

    best_loss=np.inf
    best_state=None
    best_epoch=-1
    patience=0
    hist=[]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0=time.perf_counter()

    for ep in range(1,max_epochs+1):
        model.train()
        tr_sum=0.0
        tr_n=0

        for xb,yb in train_loader:
            xb=xb.to(device,non_blocking=True)
            yb=yb.to(device,non_blocking=True)

            opt.zero_grad(set_to_none=True)

            ctx=torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=amp_enabled
            ) if torch.cuda.is_available() else nullcontext()

            with ctx:
                pred=model(xb)
                loss=loss_fn(pred,yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_sum += loss.detach().item()*len(xb)
            tr_n += len(xb)

        train_loss=tr_sum/max(tr_n,1)

        model.eval()
        va_sum=0.0
        va_n=0

        with torch.no_grad():
            for xb,yb in val_loader:
                xb=xb.to(device)
                yb=yb.to(device)

                ctx=torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=amp_enabled
                ) if torch.cuda.is_available() else nullcontext()

                with ctx:
                    pred=model(xb)
                    loss=loss_fn(pred,yb)

                va_sum += loss.detach().item()*len(xb)
                va_n += len(xb)

        val_loss=va_sum/max(va_n,1)
        sched.step(val_loss)

        hist.append({
            "epoch":ep,
            "train_loss":train_loss,
            "val_loss":val_loss,
            "lr":opt.param_groups[0]["lr"]
        })

        if val_loss < best_loss-1e-7:
            best_loss=val_loss
            best_epoch=ep
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            patience=0
        else:
            patience+=1

        if ep==1 or ep%10==0:
            print(
                f"seed={seed} B{branch} ep={ep:03d} "
                f"train={train_loss:.6f} val={val_loss:.6f}"
            )

        if patience>=cfg.EARLY_STOP_PATIENCE:
            break

    elapsed=time.perf_counter()-t0

    if best_state is None:
        raise RuntimeError("no best checkpoint")

    model.load_state_dict(best_state)
    model=model.to(device).eval()

    peak_gb=(
        torch.cuda.max_memory_allocated()/1024**3
        if torch.cuda.is_available()
        else np.nan
    )

    return model,pd.DataFrame(hist),{
        "best_epoch":best_epoch,
        "best_val_loss":best_loss,
        "train_seconds":elapsed,
        "peak_gpu_gb":peak_gb,
    }

def predict_neural(model,fd,split,branch):
    X=torch.from_numpy(
        fd[split][f"X{branch}"]
    ).float()

    loader=DataLoader(
        TensorDataset(X),
        batch_size=cfg.BATCH_SIZE,
        shuffle=False
    )

    out=[]
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb=xb.to(device)
            p=model(xb).float().cpu().numpy()
            out.append(p)

    return np.concatenate(out,axis=0)


In [ ]:

# ===== 10. 正式 blocked-CV + baseline 实验 =====
all_sample_metrics=[]
run_summary=[]
training_history=[]

def save_run_metrics(df):
    partial=pd.concat(all_sample_metrics,ignore_index=True) if all_sample_metrics else pd.DataFrame()
    if len(partial):
        partial.to_csv(
            DATA_DIR/"05_逐样本预测误差_中间保存.csv",
            index=False,encoding="utf-8-sig"
        )

for f in FOLDS:
    fold=f["fold"]
    fd=FOLD_DATA[fold]

    print("\n"+"#"*100)
    print("FOLD",fold)

    # ---------- Persistence ----------
    p1,p2=persistence_predict(fd["test"])
    met=evaluate_predictions(
        fold,"Persistence",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    # ---------- Climatology ----------
    c1=fit_climatology(fd,1)
    c2=fit_climatology(fd,2)
    p1=climatology_predict(c1,fd["test"]["target_dates"])
    p2=climatology_predict(c2,fd["test"]["target_dates"])

    met=evaluate_predictions(
        fold,"Climatology",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    # ---------- AR-EOF ----------
    ar1=fit_ar_branch(fd,1)
    ar2=fit_ar_branch(fd,2)

    pc1=ar_predict(ar1,fd,"test",1,fd["pp1"].n_modes)
    pc2=ar_predict(ar2,fd,"test",2,fd["pp2"].n_modes)
    p1=fd["pp1"].pc_to_path(pc1)
    p2=fd["pp2"].pc_to_path(pc2)

    met=evaluate_predictions(
        fold,"AR-EOF",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    run_summary.append({
        "折次":fold,
        "模型":"AR-EOF",
        "随机种子":-1,
        "分支1_AR_alpha":ar1["alpha"],
        "分支2_AR_alpha":ar2["alpha"],
        "分支1最佳epoch":np.nan,
        "分支2最佳epoch":np.nan,
        "训练秒数":0.0,
        "峰值GPU显存_GB":0.0,
        "状态":"完成"
    })

    save_run_metrics(met)

    # ---------- Neural ----------
    for name,conv_kind,rnn_kind in MODEL_SPECS:
        for seed in cfg.SEEDS:
            print("\n"+"="*88)
            print(f"fold={fold} {name} seed={seed}")

            rd=run_dir(fold,name,seed)
            result_json=rd/"运行完成.json"
            pred_npz=rd/"测试集预测结果.npz"

            if (
                (not cfg.FORCE_RETRAIN) and
                result_json.exists() and
                pred_npz.exists()
            ):
                print("[resume] 已完成，直接读取预测结果")
                info=json.loads(result_json.read_text(encoding="utf-8"))
                with np.load(pred_npz) as z:
                    p1=z["pred1"]
                    p2=z["pred2"]

                met=evaluate_predictions(
                    fold,name,seed,
                    p1,p2,
                    fd["test"]["Y1_raw"],
                    fd["test"]["Y2_raw"],
                    fd["test"]["issues"],
                    fd["test"]["target_dates"]
                )
                all_sample_metrics.append(met)
                run_summary.append(info)
                save_run_metrics(met)
                continue

            net1=build_model(name,conv_kind,rnn_kind,1,fd)
            net2=build_model(name,conv_kind,rnn_kind,2,fd)

            net1,h1,b1=train_one(
                net1,fd,1,cfg.MAX_EPOCHS_B1,seed
            )
            net2,h2,b2=train_one(
                net2,fd,2,cfg.MAX_EPOCHS_B2,seed
            )

            pc1=predict_neural(net1,fd,"test",1)
            pc2=predict_neural(net2,fd,"test",2)

            p1=fd["pp1"].pc_to_path(pc1)
            p2=fd["pp2"].pc_to_path(pc2)

            np.savez_compressed(
                pred_npz,
                pred1=p1.astype(np.float32),
                pred2=p2.astype(np.float32),
                issues=fd["test"]["issues"].astype(str),
                target_dates=fd["test"]["target_dates"].astype(str),
            )

            safe_save_model(
                net1,rd/"分支1_最佳权重.pt",
                {
                    "fold":fold,"model":name,"seed":seed,"branch":1,
                    "n_modes":fd["pp1"].n_modes
                }
            )
            safe_save_model(
                net2,rd/"分支2_最佳权重.pt",
                {
                    "fold":fold,"model":name,"seed":seed,"branch":2,
                    "n_modes":fd["pp2"].n_modes
                }
            )

            h1["折次"]=fold
            h1["模型"]=name
            h1["随机种子"]=seed
            h1["分支"]=1

            h2["折次"]=fold
            h2["模型"]=name
            h2["随机种子"]=seed
            h2["分支"]=2

            training_history += [
                h1.rename(columns={
                    "epoch":"epoch","train_loss":"train_loss",
                    "val_loss":"val_loss","lr":"lr"
                }),
                h2.rename(columns={
                    "epoch":"epoch","train_loss":"train_loss",
                    "val_loss":"val_loss","lr":"lr"
                })
            ]

            info={
                "折次":fold,
                "模型":name,
                "随机种子":seed,
                "分支1_AR_alpha":None,
                "分支2_AR_alpha":None,
                "分支1最佳epoch":int(b1["best_epoch"]),
                "分支2最佳epoch":int(b2["best_epoch"]),
                "训练秒数":float(b1["train_seconds"]+b2["train_seconds"]),
                "峰值GPU显存_GB":float(max(b1["peak_gpu_gb"],b2["peak_gpu_gb"])),
                "状态":"完成"
            }

            result_json.write_text(
                json.dumps(info,indent=2,ensure_ascii=False),
                encoding="utf-8"
            )

            run_summary.append(info)

            met=evaluate_predictions(
                fold,name,seed,
                p1,p2,
                fd["test"]["Y1_raw"],
                fd["test"]["Y2_raw"],
                fd["test"]["issues"],
                fd["test"]["target_dates"]
            )
            all_sample_metrics.append(met)

            save_run_metrics(met)

# save final raw experiment tables
sample_metrics=pd.concat(all_sample_metrics,ignore_index=True)
sample_metrics.to_csv(
    DATA_DIR/"05_逐样本预测误差.csv",
    index=False,encoding="utf-8-sig"
)

run_summary_df=pd.DataFrame(run_summary)
run_summary_df.to_csv(
    DATA_DIR/"06_训练运行记录.csv",
    index=False,encoding="utf-8-sig"
)

if training_history:
    history_df=pd.concat(training_history,ignore_index=True)
    history_df.to_csv(
        DATA_DIR/"07_神经网络训练历史.csv",
        index=False,encoding="utf-8-sig"
    )

print("blocked CV complete")


In [ ]:

# ===== 11. 汇总 Reviewer 实验结果 =====
# 每个 model / fold / seed / lead 的样本平均
metric_cols=[
    "横向误差平均值_km",
    "横向误差中位数_km",
    "横向误差RMSE_km",
    "横向误差P90_km",
    "横向误差P95_km",
    "分支1经度RMSE_度",
    "分支1经度MAE_度",
    "分支2纬度RMSE_度",
    "分支2纬度MAE_度",
]

per_run=(
    sample_metrics
    .groupby(["折次","模型","随机种子","提前期_天"],as_index=False)[metric_cols]
    .mean()
)

per_run.to_csv(
    DATA_DIR/"08_各折各随机种子各提前期指标.csv",
    index=False,encoding="utf-8-sig"
)

# 总体均值：先fold/seed run，再总体
summary=(
    per_run
    .groupby(["模型","提前期_天"],as_index=False)
    .agg(
        横向误差平均值_km=("横向误差平均值_km","mean"),
        横向误差RMSE_km=("横向误差RMSE_km","mean"),
        横向误差中位数_km=("横向误差中位数_km","mean"),
        横向误差P90_km=("横向误差P90_km","mean"),
        横向误差P95_km=("横向误差P95_km","mean"),
        横向误差RMSE标准差_km=("横向误差RMSE_km","std"),
        运行数=("横向误差RMSE_km","size"),
    )
)

# persistence skill
pers=summary.loc[
    summary["模型"]=="Persistence",
    ["提前期_天","横向误差RMSE_km"]
].rename(columns={
    "横向误差RMSE_km":"Persistence_RMSE_km"
})

summary=summary.merge(pers,on="提前期_天",how="left")
summary["相对Persistence技能评分"]=(
    1.0-summary["横向误差RMSE_km"]/summary["Persistence_RMSE_km"]
)

summary.to_csv(
    DATA_DIR/"09_各模型总体性能汇总.csv",
    index=False,encoding="utf-8-sig"
)

baseline_table=summary[
    summary["模型"].isin([
        "Persistence","Climatology","AR-EOF","Simple-LSTM"
    ])
].copy()

baseline_table.to_csv(
    DATA_DIR/"10_审稿人要求的基线模型比较.csv",
    index=False,encoding="utf-8-sig"
)

skill_table=summary[
    summary["模型"]!="Persistence"
][[
    "模型","提前期_天","横向误差RMSE_km",
    "Persistence_RMSE_km","相对Persistence技能评分"
]]

skill_table.to_csv(
    DATA_DIR/"11_相对持续性预测技能评分.csv",
    index=False,encoding="utf-8-sig"
)

# fold-wise 90d table
fold90=(
    per_run[per_run["提前期_天"]==90]
    .groupby(["折次","模型"],as_index=False)
    ["横向误差RMSE_km"].mean()
)
fold90.to_csv(
    DATA_DIR/"12_各折90天泛化误差.csv",
    index=False,encoding="utf-8-sig"
)

display(summary.sort_values(["提前期_天","横向误差RMSE_km"]))


In [ ]:

# ===== 12. 论文图：中英双版本 =====
import matplotlib.pyplot as plt

MODEL_ORDER=[
    "Persistence","Climatology","AR-EOF","Simple-LSTM",
    "EOF-Conv-LSTM","EOF-Conv-GRU","EOF-TCN-LSTM","EOF-TCN-GRU"
]

ZH_NAMES={
    "Persistence":"持续性预测",
    "Climatology":"气候态",
    "AR-EOF":"自回归EOF",
    "Simple-LSTM":"简单LSTM",
    "EOF-Conv-LSTM":"EOF-Conv-LSTM",
    "EOF-Conv-GRU":"EOF-Conv-GRU",
    "EOF-TCN-LSTM":"EOF-TCN-LSTM",
    "EOF-TCN-GRU":"EOF-TCN-GRU",
}

def save_bilingual(fig_builder,zh_name,en_name):
    use_zh_font()
    fig=fig_builder("zh")
    fig.savefig(
        FIG_ZH_DIR/(zh_name+".png"),
        dpi=cfg.FIG_DPI,bbox_inches="tight"
    )
    fig.savefig(
        FIG_ZH_DIR/(zh_name+".pdf"),
        bbox_inches="tight"
    )
    plt.close(fig)

    use_en_font()
    fig=fig_builder("en")
    fig.savefig(
        FIG_EN_DIR/(en_name+".png"),
        dpi=cfg.FIG_DPI,bbox_inches="tight"
    )
    fig.savefig(
        FIG_EN_DIR/(en_name+".pdf"),
        bbox_inches="tight"
    )
    plt.close(fig)

# ---------- 图1 严格blocked hindcast ----------
def fig1(lang):
    fig,ax=plt.subplots(figsize=(10.5,4.9))

    colors={
        "train":"#4C78A8",
        "val":"#F2A541",
        "test":"#59A14F",
    }

    y_positions=np.arange(len(FOLDS))[::-1]

    for y,f in zip(y_positions,FOLDS):
        train_start=pd.Timestamp(anchor_index["最晚目标日期"].min())
        train_end=f["train_target_end"]

        val_start=f["val_issue_start"]
        val_end=f["val_issue_end"]

        test_start=f["test_issue_start"]
        test_end=f["test_issue_end"]

        ax.plot(
            [train_start,train_end],[y,y],
            lw=12,solid_capstyle="butt",
            color=colors["train"],
            label=("训练标签可用期" if lang=="zh" else "Training-label period")
        )
        ax.plot(
            [val_start,val_end],[y,y],
            lw=12,solid_capstyle="butt",
            color=colors["val"],
            label=("验证起报期" if lang=="zh" else "Validation issue block")
        )
        ax.plot(
            [test_start,test_end],[y,y],
            lw=12,solid_capstyle="butt",
            color=colors["test"],
            label=("测试起报期" if lang=="zh" else "Test issue block")
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        [f"第{f['fold']}折" for f in FOLDS]
        if lang=="zh"
        else [f"Fold {f['fold']}" for f in FOLDS]
    )

    ax.set_xlabel("时间" if lang=="zh" else "Time")
    ax.set_title(
        "严格因果扩展窗口分块后报验证"
        if lang=="zh"
        else "Strict causal expanding-window blocked hindcast validation"
    )

    handles,labels=ax.get_legend_handles_labels()
    uniq={}
    for h,l in zip(handles,labels):
        if l not in uniq:
            uniq[l]=h
    ax.legend(
        uniq.values(),uniq.keys(),
        ncol=3,loc="upper left"
    )

    ax.grid(axis="x",alpha=.20)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig

save_bilingual(
    fig1,
    "图1_严格因果分块后报验证划分_中文",
    "Figure1_Strict_Causal_Blocked_Hindcast_English"
)

# ---------- 图2 所有模型 lead-time RMSE ----------
def fig2(lang):
    fig,ax=plt.subplots(figsize=(10.5,6.2))

    for m in MODEL_ORDER:
        q=summary[summary["模型"]==m].sort_values("提前期_天")
        if len(q)==0:
            continue
        label=ZH_NAMES[m] if lang=="zh" else m
        ax.plot(
            q["提前期_天"],
            q["横向误差RMSE_km"],
            marker="o",
            label=label
        )

    ax.set_xticks(list(cfg.LEAD_DAYS))
    ax.set_xlabel("预测提前期（天）" if lang=="zh" else "Forecast lead (days)")
    ax.set_ylabel("横向路径误差 RMSE（km）" if lang=="zh" else "Cross-track path RMSE (km)")
    ax.set_title(
        "不同模型在30/60/90天预测中的泛化误差"
        if lang=="zh"
        else "Generalization error at 30/60/90-day forecast leads"
    )
    ax.legend(fontsize=8,ncol=2)
    ax.grid(alpha=.25)
    fig.tight_layout()
    return fig

save_bilingual(
    fig2,
    "图2_不同模型不同提前期横向路径误差_中文",
    "Figure2_CrossTrack_Error_by_Forecast_Lead_English"
)

# ---------- 图3 reviewer baseline comparison ----------
def fig3(lang):
    fig,ax=plt.subplots(figsize=(11.5,6.2))

    models=MODEL_ORDER
    x=np.arange(len(models))
    width=.24

    for j,lead in enumerate(cfg.LEAD_DAYS):
        vals=[]
        for m in models:
            q=summary[
                (summary["模型"]==m)&
                (summary["提前期_天"]==lead)
            ]
            vals.append(float(q["横向误差RMSE_km"].iloc[0]) if len(q) else np.nan)

        ax.bar(
            x+(j-1)*width,
            vals,
            width,
            label=(f"{lead}天" if lang=="zh" else f"{lead} days")
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [ZH_NAMES[m] for m in models] if lang=="zh" else models,
        rotation=22,ha="right"
    )
    ax.set_ylabel("横向路径误差 RMSE（km）" if lang=="zh" else "Cross-track path RMSE (km)")
    ax.set_title(
        "与持续性、气候态、自回归EOF和简单神经网络基线的比较"
        if lang=="zh"
        else "Comparison with persistence, climatology, AR-EOF and simple neural baselines"
    )
    ax.legend()
    ax.grid(axis="y",alpha=.25)
    fig.tight_layout()
    return fig

save_bilingual(
    fig3,
    "图3_审稿人要求的基线模型比较_中文",
    "Figure3_Reviewer_Requested_Baseline_Comparison_English"
)

# ---------- 图4 persistence skill ----------
def fig4(lang):
    fig,ax=plt.subplots(figsize=(10.8,6.0))

    q=skill_table.copy()
    models=[m for m in MODEL_ORDER if m!="Persistence"]
    x=np.arange(len(models))
    width=.24

    for j,lead in enumerate(cfg.LEAD_DAYS):
        vals=[]
        for m in models:
            qq=q[(q["模型"]==m)&(q["提前期_天"]==lead)]
            vals.append(float(qq["相对Persistence技能评分"].iloc[0]) if len(qq) else np.nan)

        ax.bar(
            x+(j-1)*width,
            vals,
            width,
            label=(f"{lead}天" if lang=="zh" else f"{lead} days")
        )

    ax.axhline(0,lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [ZH_NAMES[m] for m in models] if lang=="zh" else models,
        rotation=22,ha="right"
    )
    ax.set_ylabel(
        "相对持续性预测技能评分"
        if lang=="zh"
        else "Skill relative to persistence"
    )
    ax.set_title(
        "各模型相对持续性预测的技能提升"
        if lang=="zh"
        else "Forecast skill relative to persistence"
    )
    ax.legend()
    ax.grid(axis="y",alpha=.25)
    fig.tight_layout()
    return fig

save_bilingual(
    fig4,
    "图4_相对持续性预测技能评分_中文",
    "Figure4_Skill_Relative_to_Persistence_English"
)

# ---------- 图5 fold-wise 90d heatmap ----------
def fig5(lang):
    pivot=fold90.pivot(index="模型",columns="折次",values="横向误差RMSE_km")
    pivot=pivot.reindex(MODEL_ORDER)

    fig,ax=plt.subplots(figsize=(7.5,6.3))
    im=ax.imshow(pivot.values,aspect="auto")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(
        [f"第{x}折" for x in pivot.columns]
        if lang=="zh"
        else [f"Fold {x}" for x in pivot.columns]
    )
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(
        [ZH_NAMES[x] for x in pivot.index]
        if lang=="zh"
        else list(pivot.index)
    )

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v=pivot.values[i,j]
            if np.isfinite(v):
                ax.text(j,i,f"{v:.1f}",ha="center",va="center",fontsize=8)

    cbar=fig.colorbar(im,ax=ax)
    cbar.set_label(
        "90天横向路径误差 RMSE（km）"
        if lang=="zh"
        else "90-day cross-track path RMSE (km)"
    )
    ax.set_title(
        "各分块测试折上的90天泛化性能"
        if lang=="zh"
        else "90-day generalization across blocked test folds"
    )
    fig.tight_layout()
    return fig

save_bilingual(
    fig5,
    "图5_各分块测试折90天泛化性能_中文",
    "Figure5_90Day_Generalization_Across_Blocked_Folds_English"
)

print("中文图表 ->",FIG_ZH_DIR)
print("英文图表 ->",FIG_EN_DIR)


In [ ]:


def dataframe_to_markdown_no_dependency(df, index=False):
    """Markdown table without tabulate; report generation only."""
    q = df.copy()
    if index:
        q = q.reset_index()

    def fmt(v):
        if pd.isna(v):
            return ""
        if isinstance(v, (float, np.floating)):
            return f"{float(v):.6g}"
        if isinstance(v, (pd.Timestamp, np.datetime64)):
            return str(pd.Timestamp(v))
        return str(v).replace("|", r"\|").replace("\n", " ")

    headers = [fmt(c) for c in q.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in q.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(fmt(v) for v in row) + " |")
    return "\n".join(lines)

# ===== 13. 自动生成实验结果摘要 =====
# 选总体最优hybrid：3 lead的RMSE平均
hybrids=[
    "EOF-Conv-LSTM","EOF-Conv-GRU",
    "EOF-TCN-LSTM","EOF-TCN-GRU"
]

hybrid_rank=(
    summary[summary["模型"].isin(hybrids)]
    .groupby("模型",as_index=False)
    ["横向误差RMSE_km"].mean()
    .sort_values("横向误差RMSE_km")
)

best_hybrid=str(hybrid_rank.iloc[0]["模型"])
best_hybrid_rmse=float(hybrid_rank.iloc[0]["横向误差RMSE_km"])

result_lines=[
    "# 正式训练与审稿实验结果摘要",
    "",
    f"- 最终日尺度路径数：{len(dates)}",
    f"- 180-day有效anchors：{len(anchors)}",
    f"- Blocked CV折数：{len(FOLDS)}",
    f"- 神经网络随机种子：{list(cfg.SEEDS)}",
    f"- 最优Hybrid（按30/60/90天平均横向RMSE）：{best_hybrid}",
    f"- 最优Hybrid平均横向RMSE：{best_hybrid_rmse:.3f} km",
    "",
    "## Blocked CV",
    dataframe_to_markdown_no_dependency(fold_df, index=False),
    "",
    "## 总体性能",
    dataframe_to_markdown_no_dependency(summary.sort_values(["提前期_天","横向误差RMSE_km"]), index=False),
]

(REPORT_DIR/"正式训练与审稿实验结果摘要_中文.md").write_text(
    "\n".join(result_lines),
    encoding="utf-8"
)

en_lines=[
    "# Formal retraining and reviewer experiment summary",
    "",
    f"- Daily paths: {len(dates)}",
    f"- Valid 180-day anchors: {len(anchors)}",
    f"- Blocked CV folds: {len(FOLDS)}",
    f"- Neural seeds: {list(cfg.SEEDS)}",
    f"- Best hybrid by mean 30/60/90-day cross-track RMSE: {best_hybrid}",
    f"- Mean cross-track RMSE of selected hybrid: {best_hybrid_rmse:.3f} km",
    "",
    "## Overall performance",
    dataframe_to_markdown_no_dependency(summary.sort_values(["提前期_天","横向误差RMSE_km"]), index=False),
]

(REPORT_DIR/"正式训练与审稿实验结果摘要_英文.md").write_text(
    "\n".join(en_lines),
    encoding="utf-8"
)

hybrid_rank.rename(columns={
    "模型":"模型",
    "横向误差RMSE_km":"30_60_90天平均横向误差RMSE_km"
}).to_csv(
    DATA_DIR/"13_四套Hybrid模型总体排名.csv",
    index=False,encoding="utf-8-sig"
)

print("best hybrid:",best_hybrid)
display(hybrid_rank)

In [ ]:

# ===== 14. 最终 full-data frozen model retraining =====
if not cfg.TRAIN_FINAL_FROZEN_MODEL:
    print("TRAIN_FINAL_FROZEN_MODEL=False，跳过最终模型归档。")
else:
    # 使用所有anchors；尾部target-max时间块作为early-stopping validation
    tmax=np.array([np.datetime64(a["target_max"].date()) for a in anchors])
    unique=np.array(sorted(pd.unique(tmax)))

    val_start=pd.Timestamp(unique[-cfg.FINAL_VAL_DAYS]) if len(unique)>cfg.FINAL_VAL_DAYS else pd.Timestamp(unique[int(len(unique)*.85)])

    tr_idx=np.array([
        i for i,a in enumerate(anchors)
        if pd.Timestamp(a["target_max"])<val_start
    ],dtype=int)

    va_idx=np.array([
        i for i,a in enumerate(anchors)
        if pd.Timestamp(a["target_max"])>=val_start
    ],dtype=int)

    if len(tr_idx)<cfg.MIN_TRAIN_ANCHORS or len(va_idx)<10:
        raise RuntimeError(
            f"最终模型训练划分不足: train={len(tr_idx)} val={len(va_idx)}"
        )

    fake_fold={
        "fold":999,
        "train_idx":tr_idx,
        "val_idx":va_idx,
        "test_idx":va_idx,
    }
    fd=prepare_fold(fake_fold)

    spec={x[0]:x for x in MODEL_SPECS}[best_hybrid]
    _,conv_kind,rnn_kind=spec

    final_rows=[]
    final_root=CKPT_DIR/"最终冻结模型"/best_hybrid
    final_root.mkdir(parents=True,exist_ok=True)

    for seed in cfg.SEEDS:
        print("\n"+"="*80)
        print("FINAL",best_hybrid,"seed",seed)

        sdir=final_root/f"种子{seed}"
        sdir.mkdir(parents=True,exist_ok=True)

        done=sdir/"训练完成.json"

        if done.exists() and not cfg.FORCE_RETRAIN:
            print("[resume] final frozen model already exists")
            final_rows.append(
                json.loads(done.read_text(encoding="utf-8"))
            )
            continue

        net1=build_model(best_hybrid,conv_kind,rnn_kind,1,fd)
        net2=build_model(best_hybrid,conv_kind,rnn_kind,2,fd)

        net1,h1,b1=train_one(net1,fd,1,cfg.MAX_EPOCHS_B1,seed)
        net2,h2,b2=train_one(net2,fd,2,cfg.MAX_EPOCHS_B2,seed)

        safe_save_model(
            net1,sdir/"分支1_最终权重.pt",
            {
                "model":best_hybrid,
                "seed":seed,
                "branch":1,
                "validation_start":str(val_start.date()),
            }
        )
        safe_save_model(
            net2,sdir/"分支2_最终权重.pt",
            {
                "model":best_hybrid,
                "seed":seed,
                "branch":2,
                "validation_start":str(val_start.date()),
            }
        )

        np.savez_compressed(
            sdir/"训练集预处理参数.npz",
            b1_mu=fd["pp1"].mu.astype(np.float32),
            b1_std=fd["pp1"].std.astype(np.float32),
            b1_V=fd["pp1"].V.astype(np.float32),
            b2_mu=fd["pp2"].mu.astype(np.float32),
            b2_std=fd["pp2"].std.astype(np.float32),
            b2_V=fd["pp2"].V.astype(np.float32),
            LAT1=LAT1.astype(np.float64),
            LON2=LON2.astype(np.float64),
        )

        info={
            "模型":best_hybrid,
            "随机种子":int(seed),
            "训练样本数":int(len(tr_idx)),
            "验证样本数":int(len(va_idx)),
            "验证开始日期":str(val_start.date()),
            "分支1最佳epoch":int(b1["best_epoch"]),
            "分支2最佳epoch":int(b2["best_epoch"]),
            "训练秒数":float(b1["train_seconds"]+b2["train_seconds"]),
        }
        done.write_text(
            json.dumps(info,indent=2,ensure_ascii=False),
            encoding="utf-8"
        )
        final_rows.append(info)

    pd.DataFrame(final_rows).to_csv(
        DATA_DIR/"14_最终冻结模型训练记录.csv",
        index=False,encoding="utf-8-sig"
    )

    print("最终冻结模型 ->",final_root)


In [ ]:

# ===== 15. 最终实验清单 / Handoff =====
handoff={
    "dataset":str(DATASET_PATH),
    "daily_paths":int(len(dates)),
    "date_start":str(dates.min().date()),
    "date_end":str(dates.max().date()),
    "lookback_days":cfg.LOOKBACK_DAYS,
    "lead_days":list(cfg.LEAD_DAYS),
    "valid_anchors":int(len(anchors)),
    "blocked_cv_folds":len(FOLDS),
    "validation_protocol":"strict_causal_blocked_hindcast",
    "purge_days":cfg.PURGE_DAYS,
    "neural_seeds":list(cfg.SEEDS),
    "models":[
        "Persistence","Climatology","AR-EOF",
        "Simple-LSTM","EOF-Conv-LSTM","EOF-Conv-GRU",
        "EOF-TCN-LSTM","EOF-TCN-GRU"
    ],
    "best_hybrid":best_hybrid,
    "data_directory":str(DATA_DIR),
    "figure_chinese_directory":str(FIG_ZH_DIR),
    "figure_english_directory":str(FIG_EN_DIR),
    "checkpoint_directory":str(CKPT_DIR),
}

(REPORT_DIR/"正式训练实验交接.json").write_text(
    json.dumps(handoff,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(handoff,indent=2,ensure_ascii=False))

print("\n需要交给论文绘图/写作人员的核心文件：")
for x in [
    "02_严格分块后报折次定义.csv",
    "02_严格时序数据泄漏审计.csv",
    "05_逐样本预测误差.csv",
    "08_各折各随机种子各提前期指标.csv",
    "09_各模型总体性能汇总.csv",
    "10_审稿人要求的基线模型比较.csv",
    "11_相对持续性预测技能评分.csv",
    "12_各折90天泛化误差.csv",
]:
    print(" -",DATA_DIR/x)

print("\n图表：")
print(" -",FIG_ZH_DIR)
print(" -",FIG_EN_DIR)


## FIX1: tabulate optional dependency removed

This error only affects Markdown summary generation; prior training/results remain valid. Continue from the result-summary cell.


## FIX2 最终判定标准

只有同时看到：

```text
[STRICT TEMPORAL LEAKAGE AUDIT] ALL PASS
[EOF 99% TRAIN-ONLY GATE] ALL PASS
[architecture preflight] ALL PASS
```

才把本轮结果作为 Response to Reviewer 的正式统计。

上一轮 `FINAL_RETRAIN_REVIEWER/` 保留作为 preliminary / sensitivity run；
最终论文只引用 `FINAL_RETRAIN_REVIEWER_FIX2/`。
